# Pegasus AI Workbench — LLM Configuration

Enter API keys for **FABRIC AI** and/or **NRP** to connect to LLM services.  
Other providers (Anthropic, OpenAI, Ollama, etc.) can be configured directly in OpenCode's settings.

API keys are stored locally in `~/.pegasus-ai/.env` and never leave this host.

In [ ]:
import importlib, sys, os

_nb_dir = os.path.dirname(os.path.abspath('__file__'))
for _candidate in [_nb_dir, '/opt/pegasus-ai/notebooks']:
    if _candidate not in sys.path:
        sys.path.insert(0, _candidate)

import llm_setup_helpers as helpers
importlib.reload(helpers)

import ipywidgets as widgets
from IPython.display import display, HTML

# ── Styling ──────────────────────────────────────────────────────
_field_layout = widgets.Layout(width='380px')
_btn_layout = widgets.Layout(width='80px')

# ── Banner ───────────────────────────────────────────────────────
banner = widgets.HTML(value='''
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
            color: white; padding: 20px 24px; border-radius: 10px;
            margin-bottom: 16px; font-family: system-ui, sans-serif;">
  <h2 style="margin: 0 0 6px 0; font-size: 22px;">&#9881; LLM Provider Setup</h2>
  <p style="margin: 0; opacity: 0.85; font-size: 14px;">
    Enter API keys for <b>FABRIC AI</b> and/or <b>NRP</b>.<br>
    Other providers can be configured directly in OpenCode settings.
  </p>
</div>
''')

# ── Load existing config ─────────────────────────────────────────
_existing = helpers.load_all_keys()

# ── Build per-provider key fields (FABRIC + NRP only) ────────────
_key_widgets = {}   # pid -> Password widget
_test_btns = {}     # pid -> Button
_test_outs = {}     # pid -> Output
_provider_rows = []

for pid in helpers.NOTEBOOK_PROVIDERS:
    preset = helpers.PROVIDERS[pid]
    existing_key = _existing['keys'].get(pid, '')

    key_w = widgets.Password(
        value=existing_key,
        placeholder=f'Enter {preset["name"]} API key',
        layout=_field_layout,
    )
    test_btn = widgets.Button(
        description='Test',
        button_style='info',
        icon='plug',
        layout=_btn_layout,
    )
    test_out = widgets.Output(
        layout=widgets.Layout(min_height='20px')
    )

    _key_widgets[pid] = key_w
    _test_btns[pid] = test_btn
    _test_outs[pid] = test_out

    label = widgets.HTML(
        f'<b style="font-size:14px;">{preset["name"]}</b>'
        f'<br><span style="color:#666;font-size:12px;">{preset["base_url"]}</span>'
    )

    row = widgets.VBox([
        label,
        widgets.HBox([key_w, test_btn]),
        test_out,
    ], layout=widgets.Layout(
        border='1px solid #e0e0e0', padding='10px 14px',
        margin='4px 0', border_radius='6px',
    ))
    _provider_rows.append(row)


# ── Default provider dropdown (FABRIC + NRP only) ────────────────
_default_options = [
    (helpers.PROVIDERS[pid]['name'], pid) for pid in helpers.NOTEBOOK_PROVIDERS
]
_current_default = _existing['default'] if _existing['default'] in dict(_default_options).values() else 'fabric'
default_dropdown = widgets.Dropdown(
    options=_default_options,
    value=_current_default,
    description='Default:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='350px'),
)

# ── Status area ──────────────────────────────────────────────────
status_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd', padding='10px',
        margin='10px 0', min_height='40px',
        border_radius='6px',
    )
)

save_btn = widgets.Button(
    description='Save & Apply',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='160px', height='38px'),
)


# ── Test button callbacks ────────────────────────────────────────
def _make_test_cb(pid):
    def _on_test(btn):
        out = _test_outs[pid]
        out.clear_output()
        with out:
            preset = helpers.PROVIDERS[pid]
            url = preset['base_url']
            key = _key_widgets[pid].value
            ok, msg = helpers.test_connection(url, key)
            icon = '\u2705' if ok else '\u274c'
            print(f'{icon} {msg}')
    return _on_test

for pid in _test_btns:
    _test_btns[pid].on_click(_make_test_cb(pid))


# ── Save callback ────────────────────────────────────────────────
def _on_save(btn):
    status_output.clear_output()
    with status_output:
        keys = {}
        for pid in helpers.NOTEBOOK_PROVIDERS:
            if pid in _key_widgets:
                val = _key_widgets[pid].value
                if val and val.strip():
                    keys[pid] = val

        ok, msg = helpers.save_all(
            keys=keys,
            default_provider=default_dropdown.value,
            progress_cb=lambda m: print(m),
        )
        status_output.clear_output()
        icon = '\u2705' if ok else '\u274c'
        print(f'{icon} {msg}')
        if ok:
            print()
            print('To use the AI tools:')
            print('  - OpenCode Web: click "OpenCode" in the JupyterLab Launcher')
            print('  - Terminal: open a new terminal and run: opencode')

save_btn.on_click(_on_save)


# ── Assemble UI ──────────────────────────────────────────────────
ui = widgets.VBox([
    banner,
    widgets.HTML('<h3 style="margin: 8px 0 4px 0;">API Keys</h3>'),
    widgets.HTML('<p style="color:#666;margin:0 0 8px 0;">'
                 'Enter keys for the providers you have access to.</p>'),
    *_provider_rows,
    widgets.HTML('<h3 style="margin: 14px 0 4px 0;">Default Provider</h3>'),
    widgets.HTML('<p style="color:#666;margin:0 0 4px 0;">'
                 'Used as the primary provider when launching OpenCode.</p>'),
    default_dropdown,
    widgets.HTML('<hr style="margin: 14px 0 8px 0;">'),
    save_btn,
    status_output,
])

In [ ]:
display(ui)

---
## Current Configuration

Run the cell below to see what's currently saved (keys are masked).

In [ ]:
importlib.reload(helpers)
cfg = helpers.load_all_keys()

if cfg['keys']:
    rows = ''
    for pid, key in cfg['keys'].items():
        name = helpers.PROVIDERS[pid]['name']
        masked = helpers._mask_key(key) if key else '(no key needed)'
        default_tag = ' <b style="color:#2e7d32;">[DEFAULT]</b>' if pid == cfg['default'] else ''
        rows += f'<tr><td style="padding:4px 12px 4px 0;">{name}{default_tag}</td><td style="font-family:monospace;padding:4px 0;">{masked}</td></tr>'
    display(HTML(f'''
    <div style="background: #f0f7f0; border-left: 4px solid #2e7d32;
                padding: 12px 16px; border-radius: 4px;">
      <b>Configured providers:</b>
      <table style="margin-top:8px;">{rows}</table>
    </div>
    '''))
else:
    display(HTML('''
    <div style="background: #fff3e0; border-left: 4px solid #f57c00;
                padding: 12px 16px; border-radius: 4px; font-size: 14px;">
      <b>No LLM providers configured yet.</b><br>
      Enter API keys above and click <em>Save &amp; Apply</em>.
    </div>
    '''))

---
## Using AI Tools

After saving your configuration, the following tools are ready:

### OpenCode (Web UI)
Click the **OpenCode** card in the JupyterLab **Launcher** (File > New Launcher) to open the OpenCode web interface. It reads your saved configuration and shows the configured provider and available models in the sidebar.

### Terminal
Open a terminal and run any of these:
- `opencode` — Interactive TUI coding agent
- `claude` — Claude Code CLI agent (requires Anthropic API key)

### Other Providers
To configure additional providers (Anthropic, OpenAI, Ollama, etc.), use OpenCode's built-in settings UI or edit `~/workspace/opencode.json` directly.